# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [2]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 669, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 669 (delta 83), reused 139 (delta 50), pack-reused 488 (from 1)
Receiving objects: 100% (669/669), 21.68 MiB | 37.88 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/ECE1508_GenAI


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.2 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 30 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  3%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [  6%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 13%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 16%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 20%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 23%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [6]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

20:07:24 device: cuda
20:07:24 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:07:24 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:07:33 epoch 1/20  train_loss=0.23539  val_loss=0.11650  (2.8s)
20:07:33   -> saved best checkpoint (val_loss=0.11650) to steven/outputs/patchtst_checkpoint.pt
20:07:34 epoch 2/20  train_loss=0.17142  val_loss=0.11163  (1.8s)
20:07:34   -> saved best checkpoint (val_loss=0.11163) to steven/outputs/patchtst_checkpoint.pt
20:07:36 epoch 3/20  train_loss=0.15561  val_loss=0.10409  (2.0s)
20:07:36   -> saved best checkpoint (val_loss=0.10409) to steven/outputs/patchtst_checkpoint.pt
20:07:38 epoch 4/20  train_loss=0.14294  val_loss=0.10394  (1.9s)
20:07:38   -> saved best checkpoint (val_loss=0.10394) to steven/outputs/patchtst_checkpoint.pt
20:07:40 epoch 5/20  train_loss=0.14065  val_loss=0.09824  (1.8s)
20:07:40   -> saved best checkpoint (val_loss=0.09824) to steven/outputs/patchtst_checkpoint.

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [7]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

20:08:11 device: cuda
20:08:11 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:08:11 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:08:15 epoch 1/30  beta=0.20  train_loss=0.58547 (kl=0.8114)  val_loss=0.35902 (kl=0.8001)  (2.5s)
20:08:15   -> saved best checkpoint (val_loss=0.35902) to steven/outputs/cvae_checkpoint.pt
20:08:16 epoch 2/30  beta=0.40  train_loss=0.61535 (kl=0.8001)  val_loss=0.50525 (kl=0.8000)  (1.3s)
20:08:18 epoch 3/30  beta=0.60  train_loss=0.73795 (kl=0.8000)  val_loss=0.64606 (kl=0.8000)  (1.3s)
20:08:19 epoch 4/30  beta=0.80  train_loss=0.85248 (kl=0.8000)  val_loss=0.77739 (kl=0.8000)  (1.4s)
20:08:20 epoch 5/30  beta=1.00  train_loss=0.98698 (kl=0.8000)  val_loss=0.93093 (kl=0.8000)  (1.3s)
20:08:22 epoch 6/30  beta=1.00  train_loss=0.98071 (kl=0.8000)  val_loss=0.92853 (kl=0.8000)  (1.3s)
20:08:23 epoch 7/30  beta=1.00  train_loss=0.98038 (kl=0.8000)  val_loss=0.92720 (kl=0.8000)  (1.4s)
20:08:24

## Evaluate both models on the fixed test set

In [8]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

20:08:58 device: cuda
20:08:58 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:08:58 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:08:58 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
20:08:58 evaluating on 3000 fixed test windows
20:08:59 wrote metrics to steven/outputs/metrics.json
20:08:59 overall: {
  "n_windows": 3000,
  "patchtst_reparam_mae_rmse": [
    0.09963569790124893,
    0.3773627281188965
  ],
  "cvae_reparam_mae_rmse": [
    0.16874493658542633,
    0.5140556693077087
  ],
  "patchtst_ohlc_mae_rmse": [
    2.1667176149015126,
    3.3312592210164156
  ],
  "cvae_ohlc_mae_rmse": [
    7.073854685898669,
    8.916051416770117
  ],
  "patchtst_volume_mae_rmse": [
    2025925.25,
    3514661.0
  ],
  "cvae_volume_mae_rmse": [
    3411482.75,
    4590542.0
  ],
  "patchtst_directional_accuracy": [
    0.471,
    0.5346666666666666,
    0.4806666666666

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [9]:
!python steven/src/update_report.py

20:09:03 updated steven/v1.md: results-samples, hit-summary, spread-summary, backtest-patchtst, backtest-cvae, buy-hold-benchmark
20:09:03 not auto-updated -- reread and edit by hand if the story changed: the 'In plain terms' / 'A subtle but important point' interpretation paragraphs under Results, the 'pre-retrain checkpoints' caveats in Results and Long-only backtest results, and the 'Retrain both models' checkbox under Next steps.


## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [10]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/metrics.json (deflated 83%)
  adding: steven/outputs/cvae_checkpoint.pt (deflated 9%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/samples.json (deflated 77%)
  adding: steven/outputs/sample_plots/sample0_start25969_ctx63.png (deflated 11%)
  adding: steven/outputs/sample_plots/sample1_start26935_ctx49.png (deflated 11%)
  adding: steven/outputs/sample_plots/sample2_start25277_ctx56.png (deflated 11%)
  adding: steven/outputs/sample_plots/sample3_start25032_ctx42.png (deflated 11%)
  adding: steven/outputs/sample_plots/sample4_start24873_ctx63.png (deflated 12%)
  adding: steven/outputs/patchtst_checkpoint.pt (deflated 9%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>